# Agents in LangChain

Roadmap of what to learn:

| # | Concept | Why it matters |
|---|---|---|
| 1 | What an agent is (the loop) | chain = fixed steps, agent = the LLM decides the steps |
| 2 | Tools with `@tool` | name, docstring and type hints are what the LLM sees |
| 3 | `create_agent` + `invoke` | the one function that builds an agent in LangChain v1 |
| 4 | Reading the messages | see every tool call and tool result |
| 5 | Streaming | show progress step by step instead of waiting |
| 6 | System prompt | give the agent a role and rules |
| 7 | Memory (`checkpointer` + `thread_id`) | the agent remembers earlier turns |
| 8 | Structured output (`response_format`) | get a Pydantic object back, not free text |
| 9 | Runtime context (`ToolRuntime`) | pass user id / settings to tools without the LLM seeing them |
| 10 | Tool errors | a failing tool shouldn't crash the agent |
| 11 | Built-in middleware | limits, summarization, retries without writing loop code |
| 12 | Custom middleware | your own hooks: logging, dynamic prompts, guards |
| 13 | Human in the loop | pause before risky tools and ask for approval |
| 14 | RAG agent (retriever as a tool) | the agent decides **when** to search your vector store |
| 15 | Async | for web apps / APIs |
| 16 | Common pitfalls | bugs that fail silently |

In [ ]:
from dotenv import load_dotenv
load_dotenv()   # OPENAI_API_KEY from .env

from langchain.agents import create_agent
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

## 2. Tools with `@tool`
A tool is a normal Python function. The LLM decides whether to call it from:
- the **function name** → tool name
- the **docstring** → tool description (write it for the LLM: what it does, when to use it)
- the **type hints** → argument schema

Bad docstring = the agent picks the wrong tool or never calls it.

In [ ]:
@tool
def get_weather(city: str) -> str:
    """Get the current weather for a city."""
    fake_weather = {"hyderabad": "34°C, sunny", "london": "12°C, rainy", "tokyo": "20°C, cloudy"}
    return fake_weather.get(city.lower(), "no data for this city")


@tool
def multiply(a: float, b: float) -> float:
    """Multiply two numbers. Use this for any multiplication."""
    return a * b


# what the LLM actually sees
print(get_weather.name, "-", get_weather.description)
print(multiply.args)

# tools are runnables: you can test them without an agent
print(get_weather.invoke({"city": "London"}))

get_weather - Get the current weather for a city.
{'a': {'title': 'A', 'type': 'number'}, 'b': {'title': 'B', 'type': 'number'}}
12°C, rainy


## 3. `create_agent` + `invoke`
`create_agent(model, tools)` builds the loop for you (it's a LangGraph graph under the hood).
- `model` can be a chat model object or a string like `"openai:gpt-4o-mini"`
- input and output are a **state dictionary** with a `messages` list

In [ ]:
agent = create_agent(model=llm, tools=[get_weather, multiply])

result = agent.invoke({"messages": [{"role": "user", "content": "What's the weather in Tokyo?"}]})
print(result)
print(result["messages"][-1].content)   # last message = final answer

{'messages': [HumanMessage(content="What's the weather in Tokyo?", additional_kwargs={}, response_metadata={}, id='f37160c9-95fa-4c6a-86ac-4dace2d3f12d'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 78, 'total_tokens': 92, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_2113251dd4', 'id': 'chatcmpl-EZBENFrcW6LVGVzIU6gUiYpy8eSXM', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a13eb4-32c5-7a60-b547-962f018662b8-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'Tokyo'}, 'id': 'call_pLwEt9HYsU1NZJJg9AMzZbfr', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 78, 'output_to

## 4. Reading the messages
`result["messages"]` holds the whole run. Typical order:

| Message | Meaning |
|---|---|
| `HumanMessage` | your question |
| `AIMessage` with `tool_calls` | LLM asks for a tool (no text yet) |
| `ToolMessage` | the tool's result |
| `AIMessage` with text | final answer |

The LLM can ask for **several tools at once** (parallel tool calls) when the steps don't depend on each other.

In [ ]:
result = agent.invoke({"messages": [HumanMessage("what is the temperature of hyderabad and tokyo and what is 2*3")]}) # it internally uses langgraph[loop architecture]
print(result)
for m in result["messages"]:
    m.pretty_print()

{'messages': [HumanMessage(content='what is the temperature of hyderabad and tokyo and what is 2*3', additional_kwargs={}, response_metadata={}, id='2f0bc4af-be51-404d-b614-dbc34e0acb1d'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 62, 'prompt_tokens': 89, 'total_tokens': 151, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_2113251dd4', 'id': 'chatcmpl-EZBJlH6xIEHTHAoommOvohckXGn39', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a13eb9-4afa-7753-bc3b-20b29f46f0d9-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'Hyderabad'}, 'id': 'call_wisUiLromywueihBkhOvzQAl', 'type': 'tool_call'}, {'name': 'get_weather', 'args':

## 5. Streaming
`invoke` waits for the full run. `stream` gives you each step as it happens.

| `stream_mode` | You get |
|---|---|
| `"updates"` | only what each step added (model step, tools step) |
| `"values"` | the full state after every step |
| `"messages"` | LLM tokens one by one (for chat UIs) |

In [ ]:
question = {"messages": [{"role": "user", "content": "What's 12 * 9, and is it sunny in Hyderabad?"}]}

for step in agent.stream(question, stream_mode="updates"):
    for node, update in step.items():
        print(f"--- {node} ---")
        update["messages"][-1].pretty_print()

--- model ---
================================== Ai Message ==================================
Tool Calls:
  multiply (call_BJkmAajBbntlarZeR9KjWfvM)
 Call ID: call_BJkmAajBbntlarZeR9KjWfvM
  Args:
    a: 12
    b: 9
  get_weather (call_79peIBO7x3ZZ9D170QMihOwI)
 Call ID: call_79peIBO7x3ZZ9D170QMihOwI
  Args:
    city: Hyderabad
--- tools ---
================================= Tool Message =================================
Name: multiply

108.0
--- tools ---
================================= Tool Message =================================
Name: get_weather

34°C, sunny
--- model ---
================================== Ai Message ==================================

The result of \( 12 \times 9 \) is 108. In Hyderabad, it is currently 34°C and sunny.


In [ ]:
# token by token, like ChatGPT typing
for token, metadata in agent.stream(question, stream_mode="messages"):
    if metadata["langgraph_node"] == "model" and token.content:
        print(token.content, end="", flush=True)

## 6. System prompt
`system_prompt` sets the agent's role and rules. It's sent before every LLM call.
Tell it **when** to use tools and **how** to answer.

In [ ]:
travel_agent = create_agent(
    model=llm,
    tools=[get_weather],
    system_prompt=(
        "You are a short and friendly travel assistant. "
        "Always check the weather with the tool before giving advice. "
        "Answer in at most 2 sentences."
    ),
)

result = travel_agent.invoke({"messages": [{"role": "user", "content": "Should I pack an umbrella for London?"}]})
print(result["messages"][-1].content)

Yes, you should definitely pack an umbrella for London as it's currently rainy. Enjoy your trip!


## 7. Memory: `checkpointer` + `thread_id`
By default every `invoke` starts from zero. A **checkpointer** saves the state after each step, keyed by `thread_id`.
- same `thread_id` → the agent continues the conversation
- new `thread_id` → a fresh conversation

`InMemorySaver` is lost when the kernel stops. In real apps use `SqliteSaver` / `PostgresSaver` (`langgraph-checkpoint-sqlite` / `-postgres`).

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

memory_agent = create_agent(model=llm, tools=[get_weather], checkpointer=InMemorySaver())

config = {"configurable": {"thread_id": "user-1"}}
memory_agent.invoke({"messages": [{"role": "user", "content": "Hi, I'm Harsha and I live in Hyderabad."}]}, config)

result = memory_agent.invoke({"messages": [{"role": "user", "content": "What's my name, and what's the weather where I live?"}]}, config)
print("same thread :", result["messages"][-1].content)

result = memory_agent.invoke({"messages": [{"role": "user", "content": "What's my name?"}]}, {"configurable": {"thread_id": "user-2"}})
print("new thread  :", result["messages"][-1].content)

same thread : Your name is Harsha, and the weather in Hyderabad is currently 34°C and sunny. How else can I help you?
new thread  : I don't have access to personal data about users unless it has been shared with me in the course of our conversation. Therefore, I don't know your name. If you'd like to share it, feel free!


## 8. Structured output (`response_format`)
Free text is hard to use in code. Pass a Pydantic model and the final answer comes back as that object in `result["structured_response"]`.
The agent can still call tools first; only the **final** answer is structured.

In [ ]:
from pydantic import BaseModel, Field

class TripAdvice(BaseModel):
    city: str
    temperature_c: float = Field(description="temperature in Celsius")
    pack_umbrella: bool
    tip: str = Field(description="one short travel tip")

structured_agent = create_agent(model=llm, tools=[get_weather], response_format=TripAdvice)

result = structured_agent.invoke({"messages": [{"role": "user", "content": "I'm going to London tomorrow."}]})
advice = result["structured_response"]
print(type(advice).__name__, advice)
print("umbrella?", advice.pack_umbrella)

TripAdvice city='London' temperature_c=15.0 pack_umbrella=True tip='Explore the local markets for unique finds!'
umbrella? True


## 9. Runtime context (`ToolRuntime`)
Some values must reach the tools but **not** come from the LLM: user id, API keys, DB connections, permissions.
If the LLM fills them in, it can guess or be tricked into using another user's id.

- define a `context_schema` (a dataclass)
- pass `context=...` to `invoke`
- add a `runtime: ToolRuntime` parameter to the tool → it's **hidden** from the LLM's schema

In [27]:
from dataclasses import dataclass
from langchain.tools import ToolRuntime

@dataclass     #you can also use pydantic
class UserContext:
    user_id: str

fake_orders = {"u1": ["iPhone 15", "AirPods"], "u2": ["MacBook Air"]}

@tool
def get_my_orders(runtime: ToolRuntime[UserContext]) -> str:
    """Get the current user's orders."""
    return str(fake_orders.get(runtime.context.user_id, []))

print("LLM sees args:", get_my_orders.args)   # empty: runtime is hidden

shop_agent = create_agent(model=llm, tools=[get_my_orders], context_schema=UserContext)

for uid in ["u1", "u2"]:
    result = shop_agent.invoke({"messages": [{"role": "user", "content": "What did I order?"}]}, context=UserContext(user_id=uid))
    print(uid, "->", result["messages"][-1].content)

LLM sees args: {}
u1 -> You ordered an iPhone 15 and AirPods.
u2 -> You ordered a MacBook Air.


## 10. Tool errors
What happens when a tool fails:

| Error | What the agent does |
|---|---|
| LLM sends wrong / missing arguments | error text goes back to the LLM, it retries with fixed args |
| your tool **raises** an exception | **the whole `invoke` crashes** |

So for errors the LLM can fix (bad input, not found), **return** a clear error string instead of raising,
or catch exceptions for all tools at once with a `@wrap_tool_call` middleware (more in section 12).

In [ ]:
@tool
def divide(a: float, b: float) -> float:
    """Divide a by b. Always use this tool for division."""
    if b == 0:
        raise ValueError("b must not be 0")
    return a / b

question = {"messages": [{"role": "user", "content": "Use the divide tool to compute 10 / 0."}]}

try:
    create_agent(model=llm, tools=[divide]).invoke(question)
except ValueError as e:
    print("agent crashed:", e)

In [ ]:
from langchain.agents.middleware import wrap_tool_call
from langchain_core.messages import ToolMessage

@wrap_tool_call
def tool_errors_to_llm(request, handler):
    try:
        return handler(request)   # run the real tool
    except Exception as e:
        # send the error to the LLM as the tool's result instead of crashing
        return ToolMessage(content=f"Tool error: {e}. Tell the user.", tool_call_id=request.tool_call["id"])

calc_agent = create_agent(model=llm, tools=[divide], middleware=[tool_errors_to_llm])
result = calc_agent.invoke(question)

for m in result["messages"]:
    m.pretty_print()

## 11. Built-in middleware
**Middleware** = hooks around the agent loop (before / after the model call, around each tool call).
LangChain ships common ones, you just add them to `middleware=[...]`:

| Middleware | What it does |
|---|---|
| `ModelCallLimitMiddleware` | stop runaway loops after N LLM calls |
| `ToolCallLimitMiddleware` | limit calls to all tools or one tool (e.g. an expensive search API) |
| `SummarizationMiddleware` | summarize old messages when the history gets long |
| `ModelRetryMiddleware` / `ToolRetryMiddleware` | retry with backoff on failures |
| `ModelFallbackMiddleware` | switch to a backup model if the main one fails |
| `PIIMiddleware` | redact / block emails, credit cards, etc. |
| `HumanInTheLoopMiddleware` | pause for approval (section 13) |
| `TodoListMiddleware` | gives the agent a to-do list tool for long tasks |

In [ ]:
from langchain.agents.middleware import ModelCallLimitMiddleware, SummarizationMiddleware, ToolCallLimitMiddleware

safe_agent = create_agent(
    model=llm,
    tools=[get_weather, multiply],
    checkpointer=InMemorySaver(),
    middleware=[
        ModelCallLimitMiddleware(run_limit=5),                      # max 5 LLM calls per invoke
        ToolCallLimitMiddleware(tool_name="get_weather", run_limit=2),
        SummarizationMiddleware(
            model=llm,
            trigger=("messages", 20),                                # when history reaches 20 messages...
            keep=("messages", 6),                                    # ...summarize all but the last 6
        ),
    ],
)

result = safe_agent.invoke(
    {"messages": [{"role": "user", "content": "Weather in Hyderabad, London and Tokyo?"}]},
    {"configurable": {"thread_id": "limits-demo"}},
)
print(result["messages"][-1].content)   # only 2 weather calls are allowed, the 3rd gets an error message

## 12. Custom middleware
Write your own hooks with decorators:

| Decorator | Runs | Typical use |
|---|---|---|
| `@before_model` / `@after_model` | before / after each LLM call | logging, trimming messages, guards |
| `@dynamic_prompt` | builds the system prompt per call | prompt based on user role, language, time |
| `@wrap_tool_call` | around each tool call | logging, caching, custom error handling |
| `@before_agent` / `@after_agent` | once per `invoke` | setup, final checks |

In [ ]:
from langchain.agents.middleware import ModelRequest, before_model, dynamic_prompt, wrap_tool_call

@dataclass
class RoleContext:
    role: str   # "beginner" or "expert"

@dynamic_prompt
def prompt_by_role(request: ModelRequest) -> str:
    if request.runtime.context.role == "beginner":
        return "Explain like I'm 10 years old, in 2 sentences."
    return "Answer like a senior engineer, short and technical."

@before_model # state and runtime
def log_model_call(state, runtime):
    print(f"[before_model] {len(state['messages'])} messages so far")
    return None   # return a dict to change the state, None to change nothing

@wrap_tool_call
def log_tool(request, handler):
    print(f"[tool] {request.tool_call['name']}({request.tool_call['args']})")
    return handler(request)   # run the real tool

custom_agent = create_agent(
    model=llm,
    tools=[multiply],
    context_schema=RoleContext,
    middleware=[prompt_by_role, log_model_call, log_tool],
)

for role in ["beginner", "expert"]:
    result = custom_agent.invoke(
        {"messages": [{"role": "user", "content": "What is 7 * 6, and what is multiplication?"}]},
        context=RoleContext(role=role),
    )
    print(f"{role}: {result['messages'][-1].content}\n")

## 13. Human in the loop
For risky tools (send email, delete data, payments) the agent should **pause** and wait for a person.
`HumanInTheLoopMiddleware` stops before the chosen tools; a **checkpointer is required** so the run can be resumed later.

1. `invoke` returns early with `result["__interrupt__"]` describing the pending tool call
2. you resume with `Command(resume={"decisions": [...]})`: `approve`, `edit` (change args) or `reject`

In [ ]:
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.types import Command

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an email."""
    return f"email sent to {to}"

hitl_agent = create_agent(
    model=llm,
    tools=[send_email, get_weather],
    checkpointer=InMemorySaver(),
    middleware=[HumanInTheLoopMiddleware(interrupt_on={
        "send_email": True,      # ask before sending
        "get_weather": False,    # safe, runs without asking
    })],
)

config = {"configurable": {"thread_id": "email-1"}}
result = hitl_agent.invoke(
    {"messages": [{"role": "user", "content": "Email bob@example.com that the meeting moved to 4pm."}]},
    config,
)

# the agent stopped before send_email
pending = result["__interrupt__"][0].value["action_requests"][0]
print("waiting for approval:", pending["name"], pending["args"])

In [ ]:
# approve it (other options: {"type": "reject", "message": "why"} or {"type": "edit", "edited_action": {...}})
result = hitl_agent.invoke(Command(resume={"decisions": [{"type": "approve"}]}), config)
print(result["messages"][-1].content)

## 14. RAG agent: retriever as a tool
A RAG **chain** always searches, even for "hi". A RAG **agent** decides **if** and **what** to search, and can search again with a better query.
`create_retriever_tool` turns any retriever (from the VectorStore notes) into a tool.

In [25]:
from langchain_core.documents import Document
from langchain_core.tools import create_retriever_tool
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings

policy_docs = [
    Document(page_content="Employees get 24 paid leaves per year.", metadata={"source": "hr_policy.txt"}),
    Document(page_content="Work from home is allowed 2 days a week.", metadata={"source": "hr_policy.txt"}),
    Document(page_content="Laptops are replaced every 3 years.", metadata={"source": "it_policy.txt"}),
]
store = InMemoryVectorStore.from_documents(policy_docs, OpenAIEmbeddings())

policy_tool = create_retriever_tool(
    store.as_retriever(search_kwargs={"k": 2}),
    name="search_company_policy",
    description="Search the company HR and IT policies. Use for any question about leaves, WFH or laptops.",
)

rag_agent = create_agent(
    model=llm,
    tools=[policy_tool],
    system_prompt="Answer only from the policy search results. If they don't contain the answer, say you don't know.",
)

for q in ["Hi!", "How many leaves do I get, and how often is my laptop replaced?"]:
    result = rag_agent.invoke({"messages": [{"role": "user", "content": q}]})
    used = [call["args"] for m in result["messages"] for call in getattr(m, "tool_calls", [])]
    print(f"Q: {q}\n   searches: {used}\n   A: {result['messages'][-1].content}\n")

Q: Hi!
   searches: []
   A: Hello! How can I assist you today?

Q: How many leaves do I get, and how often is my laptop replaced?
   searches: [{'query': 'leave policy'}, {'query': 'laptop replacement policy'}]
   A: You get 24 paid leaves per year, and your laptop is replaced every 3 years.



## 15. Async
Every method has an async version: `ainvoke`, `astream`. Tools can be `async def` too.
Use them in async apps (FastAPI, async LangGraph) so one request doesn't block others.

In [ ]:
result = await agent.ainvoke({"messages": [{"role": "user", "content": "What is 3.5 * 8?"}]})
print(result["messages"][-1].content)

async for step in agent.astream({"messages": [{"role": "user", "content": "Weather in Tokyo?"}]}, stream_mode="updates"):
    print(list(step.keys()))

## 16. Common pitfalls
1. **Vague docstrings** → the agent picks the wrong tool or never uses it. Describe what the tool does **and when** to use it.
2. **Forgetting `thread_id`** with a checkpointer → error, or every call shares one conversation. One thread per user / chat.
3. **No checkpointer for human in the loop** → the agent can't pause and resume.
4. **Secrets as tool arguments** → the LLM can see / invent them. Use `ToolRuntime` context instead.
5. **No call limits** → a confused agent can loop and burn tokens. Add `ModelCallLimitMiddleware`.
6. **Too many tools** → accuracy drops past ~10-20 tools. Split into smaller agents, or use `LLMToolSelectorMiddleware`.
7. **Tools that raise** → the whole run crashes. Return error strings or catch them with `@wrap_tool_call`.
8. **Huge tool outputs** → fill the context window. Return only what the LLM needs (top rows, short summaries).
9. **Using an agent when a chain is enough** → if the steps are always the same, a chain is cheaper, faster and more predictable.

### Summary

| Need | Use |
|---|---|
| LLM decides which tools to call | `create_agent(model, tools)` |
| remember earlier turns | `checkpointer` + `thread_id` |
| typed result for your code | `response_format=PydanticModel` |
| user id / secrets in tools | `context_schema` + `ToolRuntime` |
| limits, summaries, retries | built-in middleware |
| your own logic around the loop | `@before_model`, `@dynamic_prompt`, `@wrap_tool_call` |
| approval before risky actions | `HumanInTheLoopMiddleware` + `Command(resume=...)` |
| search your docs only when needed | `create_retriever_tool` |

### Next to explore
Multi-agent systems (an agent used as a tool of another agent), and custom graphs in LangGraph when `create_agent`'s loop isn't flexible enough.